In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pickle
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
graph_embeddings = torch.load(
    "/content/drive/MyDrive/ds003844_processed/spatial_graph_embeddings.pt",
    map_location=device
)
print(graph_embeddings.shape)

In [ ]:
with open(
    "/content/drive/MyDrive/ds003844_processed/recording_metadata.pkl",
    "rb"
) as f:
    metadata = pickle.load(f)
sequence_lengths = metadata["sequence_lengths"]
recording_names = metadata["recording_names"]
channel_counts = metadata["channel_counts"]

In [ ]:
sequences = []
idx = 0
for length in sequence_lengths:
    sequences.append(
        graph_embeddings[idx:idx + length]
    )
    idx += length

In [ ]:
print("Number of sequences:", len(sequences))
for i in range(5):
    print(f"Recording {i+1}: {sequences[i].shape}")
reconstructed = torch.cat(sequences, dim=0)
print("\nReconstruction Correct:",torch.allclose(reconstructed, graph_embeddings))

In [ ]:
from torch.utils.data import Dataset
class BrainSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return self.sequences[idx]

In [ ]:
dataset = BrainSequenceDataset(sequences)
print(len(dataset))

In [ ]:
max_len = max(sequence_lengths)
print("Maximum sequence length:", max_len)

In [ ]:
from torch.nn.utils.rnn import pad_sequence
padded_sequences = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=0.0
)
print(padded_sequences.shape)

In [ ]:
attention_mask = torch.zeros(
    len(sequence_lengths),
    max_len,
    dtype=torch.bool
)
for i, length in enumerate(sequence_lengths):
    attention_mask[i, :length] = True

In [ ]:
print(attention_mask.shape)
print(attention_mask[0])

In [ ]:
import math
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
positional_encoding = PositionalEncoding(
    d_model=128,
    max_len=max_len
).to(device)
x = positional_encoding(padded_sequences)
print(x.shape)

In [ ]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=128,
    nhead=8,
    dim_feedforward=512,
    dropout=0.1,
    batch_first=True
)

transformer = nn.TransformerEncoder(
    encoder_layer,
    num_layers=2
).to(device)

In [ ]:
src_padding_mask = (~attention_mask).to(device)
x = positional_encoding(padded_sequences)
embeddings = transformer(
    x,
    src_key_padding_mask=src_padding_mask
)
print(embeddings.shape)

In [ ]:
reconstruction_head = nn.Linear(
    128,
    128
).to(device)

In [ ]:
reconstructed = reconstruction_head(embeddings)
print(reconstructed.shape)

In [ ]:
attention_mask = attention_mask.to(device)
mask_ratio = 0.15
random_mask = (
    torch.rand(
        padded_sequences.shape[:2],
        device=device
    ) < mask_ratio
)
# Never mask padding positions
random_mask &= attention_mask
print(random_mask.shape)
print(random_mask.sum())

In [ ]:
masked_input = padded_sequences.clone()
masked_input[random_mask] = 0.0

In [ ]:
idx = random_mask.nonzero()[0]
print("Masked position:", idx)
print("\nOriginal embedding:")
print(padded_sequences[idx[0], idx[1], :5])
print("\nMasked embedding:")
print(masked_input[idx[0], idx[1], :5])

In [ ]:
x = positional_encoding(masked_input)
embeddings = transformer(
    x,
    src_key_padding_mask=~attention_mask
)
reconstructed = reconstruction_head(embeddings)
print(reconstructed.shape)

In [ ]:
criterion = nn.MSELoss()
loss = criterion(
    reconstructed[random_mask],
    padded_sequences[random_mask]
)
print(loss)

In [ ]:
optimizer = torch.optim.Adam(
    list(transformer.parameters()) +
    list(reconstruction_head.parameters()),
    lr=1e-4
)

In [ ]:
epochs = 1000
for epoch in range(epochs):
    transformer.train()
    reconstruction_head.train()
    # Random masking every epoch
    random_mask = (
        torch.rand(
            padded_sequences.shape[:2],
            device=device
        ) < 0.15
    )
    random_mask &= attention_mask
    masked_input = padded_sequences.clone()
    masked_input[random_mask] = 0.0
    optimizer.zero_grad()
    x = positional_encoding(masked_input)
    embeddings = transformer(
        x,
        src_key_padding_mask=~attention_mask
    )
    reconstructed = reconstruction_head(embeddings)
    loss = criterion(
        reconstructed[random_mask],
        padded_sequences[random_mask]
    )
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {loss.item():.6f}"
        )

In [ ]:
transformer.eval()
with torch.no_grad():
    x = positional_encoding(padded_sequences)
    spatio_temporal_embeddings = transformer(
        x,
        src_key_padding_mask=~attention_mask
    )

In [ ]:
final_embeddings = []
for i, length in enumerate(sequence_lengths):
    final_embeddings.append(
        spatio_temporal_embeddings[i, :length]
    )
final_embeddings = torch.cat(
    final_embeddings,
    dim=0
)
print(final_embeddings.shape)

In [ ]:
torch.save(
    final_embeddings.cpu(),
    "/content/drive/MyDrive/ds003844_processed/spatio_temporal_embeddings.pt"
)

In [ ]:
torch.save(
    transformer.state_dict(),
    "/content/drive/MyDrive/ds003844_processed/masked_transformer.pt"
)

In [ ]:
metadata = {
    "sequence_lengths": sequence_lengths,
    "recording_names": recording_names,
    "channel_counts": channel_counts
}
with open(
    "/content/drive/MyDrive/ds003844_processed/spatio_temporal_metadata.pkl",
    "wb"
) as f:
    pickle.dump(metadata, f)